# Tutorial 4 — Knowledge Base and Learning

A **knowledge base** (KB) is a set of closed formulas whose satisfaction we want to maximise.
`KB([φ1, φ2, …])` evaluates every clause and aggregates their truth values into a single one with the `Logic`'s `SatAgg` (pMeanError, p = 2, by default).
The training objective is simply `loss = 1 − sat`.

**Key pattern**
```python
kb = KB([sig.parse("∀ x_pos: InDisc(x_pos)"),          # parse once, outside the loop
             sig.parse("∀ x_neg: ¬InDisc(x_neg)")])
for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
    interp["x_pos"], interp["x_neg"] = x_pos_batch, x_neg_batch   # ground the batch
    sat = interp(kb).value
    (1.0 - sat).backward()
```

This tutorial learns a binary classifier `InDisc` from the knowledge base
$$\mathcal{K} = \{\,\forall x^+\, \mathrm{InDisc}(x^+),\quad \forall x^-\, \lnot \mathrm{InDisc}(x^-)\,\}$$
"every positive point is in the disc, and no negative point is". It then shows the `kb_backward` training helpers, derived predicates with `sig.define`, and the evaluation cache.

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sltn
from sltn import Signature, Interpretation, KB
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1 — Dataset

200 points in $[0,1]^2$. A point is **positive** if it is inside the disc of radius 0.3 centred on $(0.5, 0.5)$. 160 points are used for training and 40 for testing.

In [ ]:
n = 200
pts    = torch.rand(n, 2)
labels = (pts - 0.5).pow(2).sum(1) < 0.09   # True = inside the disc

n_train = int(n * 0.8)
pts_tr, lbl_tr = pts[:n_train], labels[:n_train]
pts_te, lbl_te = pts[n_train:], labels[n_train:]
print(f"training: {lbl_tr.sum().item()} positive, {(~lbl_tr).sum().item()} negative")

plt.figure(figsize=(4, 4))
plt.scatter(*pts_tr[lbl_tr].T,  label="positive", c="C0", alpha=0.6, s=15)
plt.scatter(*pts_tr[~lbl_tr].T, label="negative", c="C1", alpha=0.6, s=15)
plt.legend(); plt.title("Training data"); plt.tight_layout(); plt.show()

## 2 — Signature, network and knowledge base

The signature has one sort, `Point`, two variables for the two classes, and the predicate `InDisc`, grounded by a network. The formulas are parsed once, outside the training loop: they are syntax trees, with no data.

> **Avoid batch normalisation in predicate networks.** Each atom calls its network on its own batch: here `InDisc(x_pos)` sees only positives and `InDisc(x_neg)` only negatives. In training mode, batch norm normalises each call by that batch's own statistics, so the network can no longer tell the two classes apart. With two `BatchNorm1d` layers added to `InDiscNet`, the test accuracy drops from about 0.98 to 0.62–0.77. Use layer normalisation, or none.

In [ ]:
sig = Signature("disc")
sig.sort("Point")
sig.variable("x_pos", "Point")
sig.variable("x_neg", "Point")
sig.predicate("InDisc", ["Point"])

interp = Interpretation(sig).to(device)
interp["Point"] = sltn.Type("R2", shape=(2,))


class InDiscNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ELU(),
            nn.Linear(16, 16), nn.ELU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )

    def forward(self, a):                 # a: [B, 2]
        return self.net(a)                # [B, 1]


interp["InDisc"] = InDiscNet()            # moved to the interpretation's device

phi_pos = sig.parse("forall x_pos: InDisc(x_pos)")
phi_neg = sig.parse("forall x_neg: not InDisc(x_neg)")
kb = KB([phi_pos, phi_neg])
print(kb)

### `KB([...])` and the `{ φ1, φ2, … }` syntax

`KB([phi_pos, phi_neg])` and `sig.parse("{ φ1, φ2 }")` build the same knowledge base. The brace form is convenient when the whole KB is written as one string.

**The clauses must be closed** (no free variables). Both forms check it when the KB is built, and raise a `ValueError` for an open clause, instead of failing later with a shape error during evaluation.

In [ ]:
kb_brace = sig.parse("{ forall x_pos: InDisc(x_pos), forall x_neg: not InDisc(x_neg) }")
print("Same clauses:", [str(c) for c in kb_brace.clauses] == [str(c) for c in kb.clauses])

# An open clause (x_pos is free) is rejected by both forms
for build in [lambda: sig.parse("{ InDisc(x_pos) }"), lambda: KB([sig.parse("InDisc(x_pos)")])]:
    try:
        build()
    except ValueError as e:
        print("ValueError:", e)

## 3 — Training

Each class has its own `DataLoader`, and each step grounds `x_pos` with a batch of positives and `x_neg` with a batch of negatives. `zip` pairs the batches and stops at the shorter loader, so both loaders are given the same number of batches per epoch (4): every point is then used once per epoch. There are more negatives (104) than positives (56), so the negative batches are larger (26 vs 14). That doesn't make the negatives weigh more: each `∀` averages over its own batch, and `SatAgg` combines the two clauses with equal weight.

In [ ]:
N_BATCHES = 4
pos_tr, neg_tr = pts_tr[lbl_tr], pts_tr[~lbl_tr]
loader_pos = DataLoader(TensorDataset(pos_tr), batch_size=math.ceil(len(pos_tr) / N_BATCHES), shuffle=True)
loader_neg = DataLoader(TensorDataset(neg_tr), batch_size=math.ceil(len(neg_tr) / N_BATCHES), shuffle=True)
print("batches per epoch:", len(loader_pos), len(loader_neg))

opt = torch.optim.Adam(interp.parameters(), lr=1e-2)

EPOCHS = 200
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
        opt.zero_grad()
        interp["x_pos"] = x_pos_batch.to(device)
        interp["x_neg"] = x_neg_batch.to(device)
        sat  = interp(kb).value
        loss = 1.0 - sat
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    if epoch % 25 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:4d}  mean loss {epoch_loss / N_BATCHES:.4f}")

### 3b — The `kb_describe` / `kb_backward` training helpers

`sltn.tools.optim` provides helpers for the same loop:

- `kb_describe(kb)` names the clauses `φ_0`, `φ_1`, … for logging, prints them, and returns a dict to reuse.
- `kb_backward(optimizer, interp, kb_dict)` evaluates every clause and backpropagates `1 − SatAgg(...)`. With the default `aggregator=None` (or `"satagg"`) this is exactly the loss of the loop above. It raises an error if the KB depends on trainable parameters that neither the optimizer nor the interpretation holds, such as a network used inside a lambda (Tutorial 1, 2.4).
- With another `aggregator` (`"mgda"`, `"pcgrad"`, `"upgrad"`, …), each clause is a separate objective and their gradients are combined with [TorchJD](https://torchjd.org/stable/), a library for multi-objective optimisation by Jacobian descent ([paper](https://arxiv.org/abs/2406.16232)). It needs the optional `sltn[multiobjective]` extra and is not shown here.
- `kb_eval(interp, kb_dict)` evaluates the clauses under `torch.no_grad()`, for monitoring. The loop below uses it to print the satisfaction of each clause on the whole training set, so you can see which clause lags behind. (`kb_backward` also returns each clause's satisfaction on the current batch, in `stats["sats"]`, but one batch is noisy.)

Like `interp(phi)`, `kb_backward` uses whatever is grounded in `interp`, so ground the batch first. Below, a second network is trained this way.

In [ ]:
from sltn import kb_backward, kb_describe, kb_eval

interp_helper = Interpretation(sig).to(device)
interp_helper["Point"] = sltn.Type("R2", shape=(2,))
interp_helper["InDisc"] = InDiscNet()

kb_dict = kb_describe(kb)                     # φ_0, φ_1 (prints once)
opt_helper = torch.optim.Adam(interp_helper.parameters(), lr=1e-2)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
        opt_helper.zero_grad()
        interp_helper["x_pos"] = x_pos_batch.to(device)
        interp_helper["x_neg"] = x_neg_batch.to(device)
        stats = kb_backward(opt_helper, interp_helper, kb_dict)
        opt_helper.step()
        epoch_loss += stats["agg_loss"]
    if epoch % 25 == 0 or epoch == EPOCHS - 1:
        # Satisfaction of each clause on the whole training set (no gradient)
        interp_helper["x_pos"], interp_helper["x_neg"] = pos_tr.to(device), neg_tr.to(device)
        sats = kb_eval(interp_helper, kb_dict)
        print(f"epoch {epoch:4d}  mean loss {epoch_loss / N_BATCHES:.4f}   "
              + "  ".join(f"{name} {sat:.3f}" for name, sat in sats.items()))

# The parameters, named after the symbol they ground
print("\nnamed_parameters():")
for name, p in interp_helper.named_parameters():
    print(f"  {name:<22s} shape={tuple(p.shape)}")

## 4 — Evaluation

The accuracy of both networks on the 40 test points (a point is classified as positive when `InDisc` > 0.5), and the learned `InDisc` on a dense grid, to see the decision boundary. The grid is ground to a new variable, `x_grid`: a variable can be declared at any time, and the formula `InDisc(x_grid)` has one truth degree per grid point.

In [ ]:
def accuracy(model_interp):
    with torch.no_grad():
        pred = model_interp["InDisc"](pts_te.to(device))[:, 0].cpu() > 0.5
    return (pred == lbl_te).float().mean().item()

print(f"test accuracy:  manual loop {accuracy(interp):.2f}   kb_backward {accuracy(interp_helper):.2f}")

g = torch.linspace(0, 1, 100)
gx, gy = torch.meshgrid(g, g, indexing="ij")
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1).to(device)

sig.variable("x_grid", "Point")
interp["x_grid"] = grid
with torch.no_grad():
    sat = interp("InDisc(x_grid)").value.cpu().numpy()

plt.figure(figsize=(4, 4))
plt.contourf(gx.numpy(), gy.numpy(), sat.reshape(100, 100), levels=20, cmap="RdYlGn")
plt.colorbar(label="InDisc(x)")
plt.scatter(*pts_te[lbl_te].T,  c="blue", s=15, label="positive (test)")
plt.scatter(*pts_te[~lbl_te].T, c="red",  s=15, label="negative (test)")
plt.legend(fontsize=8); plt.title("Learned InDisc(x)"); plt.tight_layout(); plt.show()

## 5 — Derived predicates (or terms) with `sig.define`

`sig.define(name, params, body)` declares a **macro**: whenever `sig.parse(...)` meets `name(t1, …, tn)`, it replaces it with `body`, where each parameter `params[i]` is replaced by `ti`. This happens at parse time, before evaluation, so a defined name needs **no grounding**: its body uses symbols that are already grounded. A body can be a formula (a derived predicate) or a term (a derived function).

```python
sig.define("LowerRight", ["z"], "IsRight(z) & IsLow(z)")
sig.parse("LowerRight(p)")              # becomes IsRight(p) & IsLow(p)
```

The cell below builds a small signature with two fixed predicates on points, `IsRight` (first coordinate > 0.5) and `IsLow` (second coordinate < 0.5), defines `LowerRight`, and checks that it gives the same truth values as its expansion.

In [ ]:
sig2 = Signature("quadrants")
sig2.sort("Point")
sig2.variable("p", "Point")
sig2.predicate("IsRight", ["Point"])
sig2.predicate("IsLow", ["Point"])
sig2.define("LowerRight", ["z"], "IsRight(z) & IsLow(z)")   # a macro: no grounding needed

interp2 = Interpretation(sig2)
interp2["Point"] = sltn.Type("R2", shape=(2,))
interp2["IsRight"] = lambda a: (a[:, [0]] > 0.5).float()
interp2["IsLow"]   = lambda a: (a[:, [1]] < 0.5).float()
interp2["p"] = torch.rand(6, 2)

print("parsed:", sig2.parse("LowerRight(p)"))
with torch.no_grad():
    defined  = interp2("LowerRight(p)").value[:, 0]
    expanded = interp2("IsRight(p) & IsLow(p)").value[:, 0]

print("\n      point       LowerRight(p)   IsRight(p) & IsLow(p)")
for pt, d, e in zip(interp2["p"], defined, expanded):
    print(f"  [{pt[0]:.2f}, {pt[1]:.2f}]   {d:13.2f}   {e:21.2f}")

## 6 — The evaluation cache

During one call `interp(phi)`, every sub-formula and sub-term is evaluated **once** and its result reused wherever it appears again in the same context: the key is the sub-formula's text and the groundings of its variables. A sub-formula under a diagonal quantifier, for example, is a different context from the same text outside it, so it is evaluated again.

- The cache lives only for one call: it is created at the start of `interp(phi)` and discarded at the end, so new data is always used.
- It is on by default (`interp.use_cache = True`). The values and gradients are the same with it on or off; only the number of calls changes. (One exception: with dropout in training mode, a repeated atom shares one dropout mask, which keeps the formula consistent: `InDisc(x) <-> InDisc(x)` is exactly 1.)

The cell counts how many times the predicate is called for a formula that repeats it three times.

In [ ]:
calls = {"IsRight": 0}

class CountedIsRight(nn.Module):
    def forward(self, a):
        calls["IsRight"] += 1
        return (a[:, [0]] > 0.5).float()

interp2["IsRight"] = CountedIsRight()
formula = "IsRight(p) & IsRight(p) & IsRight(p)"

for use_cache in [True, False]:
    interp2.use_cache = use_cache
    calls["IsRight"] = 0
    with torch.no_grad():
        interp2(formula)
    print(f"use_cache={use_cache}:  IsRight called {calls['IsRight']} time(s) for {formula}")

interp2.use_cache = True

---

**Next:** [Tutorial 5 — Structural LTN](5-StructuralLTN.ipynb) extends everything covered so far with structural axes for reasoning over time, sequences, and graphs.